# 01 · Exploratory Data Analysis — MovieLens-1M

**Goal:** understand the rating, user, item, and temporal structure of
MovieLens-1M *before* any modelling, and use the findings to **motivate**
every design choice in the recall stage (rating ≥ 4 binarisation, min-5
user filter, popularity vs content channels, sequential modelling).

Each plot is also exported to `experiments/results/eda/*.png` so it can
be embedded directly in the README without checking notebook outputs
into git.

## Notebook map

| § | What | Motivates… |
|---|---|---|
| 3 | Rating distribution (1–5) | `rating_threshold = 4` cutoff |
| 4 | User activity CDF | `min_interactions_per_user = 5` filter |
| 5 | Item popularity (Zipf) | popularity-bias debias + cold-start channel |
| 6 | Temporal density | sequential modelling (SASRec) |
| 7 | Genre frequency | content-based cold-start TF-IDF features |
| 8 | Cold-start sub-population | quantify what the min-5 filter discards |
| 9 | Long-tail coverage (Lorenz / Gini) | budget for re-ranking debias |

## 1. Setup

In [ ]:
import json
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

ROOT = Path('.').resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
os.chdir(ROOT)
print('working dir:', ROOT)

FIG_DIR = ROOT / 'experiments' / 'results' / 'eda'
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 150,
    'figure.figsize': (8, 5),
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.3,
    'font.size': 11,
})

def save_fig(fig, name):
    out = FIG_DIR / name
    fig.savefig(out, bbox_inches='tight')
    print(f'saved {out.relative_to(ROOT)}')

## 2. Load raw and processed data

* `ratings.dat` — *raw* 1 M ratings (all 1–5 scores, before any filter)
* `interactions.parquet` — positives only (rating ≥ 4) after dense re-indexing
* `item_features.parquet`, `user_features.parquet` — engineered features
* `id_maps.json` — genre id ↔ name lookup

In [ ]:
RAW = ROOT / 'data' / 'raw' / 'movielens_1m' / 'ml-1m'
PROC = ROOT / 'data' / 'processed' / 'movielens_1m'

raw_ratings = pd.read_csv(
    RAW / 'ratings.dat', sep='::', engine='python', header=None,
    names=['raw_user', 'raw_item', 'rating', 'ts'],
)
print(f'raw ratings: {len(raw_ratings):,} rows, {raw_ratings.raw_user.nunique():,} users, {raw_ratings.raw_item.nunique():,} items')

inter = pd.read_parquet(PROC / 'interactions.parquet')
items = pd.read_parquet(PROC / 'item_features.parquet')
users = pd.read_parquet(PROC / 'user_features.parquet')
id_maps = json.loads((PROC / 'id_maps.json').read_text())
GENRE_NAME = {v: k for k, v in id_maps['genre_map'].items()}

print(f'processed positives: {len(inter):,} rows, {inter.user_id.nunique():,} users, {inter.item_id.nunique():,} items')
print(f'survival rate: {len(inter) / len(raw_ratings):.1%} of raw ratings kept')

## 3. Rating distribution — why `rating ≥ 4`?

MovieLens uses 1–5 stars. The most common modelling choice for implicit
feedback is to binarise: 1 if the user *liked* the movie, 0 otherwise.
We need a defensible cutoff.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

counts = raw_ratings['rating'].value_counts().sort_index()
pct = counts / counts.sum()
colors = ['#d9d9d9'] * 3 + ['#2b8cbe', '#08589e']  # 4/5 highlighted
axes[0].bar(counts.index, counts.values, color=colors, edgecolor='black', linewidth=0.5)
for x, v, p in zip(counts.index, counts.values, pct):
    axes[0].text(x, v + 8000, f'{p:.1%}', ha='center', fontsize=10)
axes[0].set_xticks([1, 2, 3, 4, 5])
axes[0].set_xlabel('rating (stars)')
axes[0].set_ylabel('count')
axes[0].set_title('Rating distribution (raw)')
axes[0].axvspan(3.5, 5.5, alpha=0.08, color='#08589e')
axes[0].text(4.5, counts.values.max() * 0.92, 'kept as positives', ha='center', fontsize=10, color='#08589e')

cum = pct[::-1].cumsum()[::-1]
axes[1].plot(cum.index, cum.values, marker='o', linewidth=2, color='#08589e')
axes[1].axhline(pct[4] + pct[5], color='red', linestyle='--', alpha=0.6, label=f'≥4 keeps {pct[4]+pct[5]:.1%}')
axes[1].set_xlabel('threshold (keep rating ≥ T)')
axes[1].set_ylabel('fraction of raw ratings kept')
axes[1].yaxis.set_major_formatter(PercentFormatter(1.0))
axes[1].set_xticks([1, 2, 3, 4, 5])
axes[1].set_title('Survival vs binarisation threshold')
axes[1].legend()

fig.suptitle('Figure 1 — Rating distribution motivates rating ≥ 4 cutoff', fontsize=12, y=1.02)
fig.tight_layout()
save_fig(fig, '01_rating_distribution.png')
plt.show()

print()
print(f'rating ≥ 4 keeps {pct[4]+pct[5]:.1%} of raw ratings — a strong positive signal, not noise.')
print(f'rating ≥ 3 would keep {pct[3]+pct[4]+pct[5]:.1%} but adds lukewarm "watched" signal — too much label noise.')

## 4. User activity — distribution shape

MovieLens-1M is **pre-filtered** by the dataset authors to ≥20 ratings
per user, so almost every user already meets our `min_interactions ≥ 5`
filter on positives alone. The interesting story here is therefore not
*how many* we lose, but **the wide spread of activity** — from ~10 to
~2 000 positives per user. That spread is what makes hard-tail users
(bottom decile, ~25 positives) qualitatively different from
power-users (top decile, ~500+) — quantified in §8.

In [ ]:
user_act = raw_ratings.groupby('raw_user').size().rename('n_ratings')
user_act_pos = raw_ratings[raw_ratings['rating'] >= 4].groupby('raw_user').size().rename('n_positives')
act = pd.concat([user_act, user_act_pos], axis=1).fillna(0).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(13, 6.5))

ax = axes[0]
sorted_n = np.sort(act['n_positives'].values)
cdf = np.arange(1, len(sorted_n) + 1) / len(sorted_n)
ax.plot(sorted_n, cdf, linewidth=2, color='#08589e')
thr_label_x = {5: 5, 20: 16, 50: 62, 200: 200}  # nudge 20/50 apart on log-x
for thr in [5, 20, 50, 200]:
    frac = (act['n_positives'] < thr).mean()
    ax.axvline(thr, color='red', alpha=0.3, linestyle='--')
    ax.text(thr_label_x[thr], 1.02, f'<{thr}: {frac:.1%}', ha='center', fontsize=9, rotation=0)
ax.set_xscale('log')
ax.set_xlabel('#positive interactions per user (log)')
ax.set_ylabel('CDF (fraction of users)')
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_title('User activity CDF (positives only)')

ax = axes[1]
buckets = ['<10', '10-29', '30-99', '100-299', '300-999', '1000+']
edges = [0, 9, 29, 99, 299, 999, np.inf]
bucket = pd.cut(act['n_positives'], bins=edges, labels=buckets, include_lowest=True)
counts = bucket.value_counts().reindex(buckets)
colors = ['#fc8d59', '#fdbb84', '#a6cee3', '#3182bd', '#08519c', '#08306b']
ax.bar(buckets, counts.values, color=colors, edgecolor='black', linewidth=0.5)
for i, c in enumerate(counts.values):
    ax.text(i, c + 30, f'{c:,}\n({c/counts.sum():.0%})', ha='center', fontsize=9)
ax.set_ylim(0, counts.max() * 1.22)
ax.set_ylabel('#users')
ax.set_xlabel('positives per user (bucketed)')
ax.set_title('User population by activity bucket')

fig.suptitle('Figure 2 — Activity ranges over 2+ orders of magnitude', fontsize=12, y=1.02)
fig.tight_layout()
save_fig(fig, '02_user_activity.png')
plt.show()

print()
print(f'positives per user — min: {act["n_positives"].min()}, median: {act["n_positives"].median():.0f}, max: {act["n_positives"].max()}')
print(f'cv (std / mean): {act["n_positives"].std() / act["n_positives"].mean():.2f} — wide spread')
print()
print('ML-1M is pre-filtered to ≥20 raw ratings/user by the dataset authors;')
print('after rating≥4 binarisation, only 6 users (0.1%) have <5 positives — they are dropped by min_interactions=5 as a safety belt.')
print('The real story is the **wide activity spread** — bottom-decile users behave very differently from top-decile (see §8).')

## 5. Item popularity — Zipf log-log

Long-tail distribution is the foundational assumption behind almost every
design choice in recsys: popularity bias, debias re-ranking, content-based
cold-start. We need to confirm ML-1M really *is* Zipf, and quantify the
head/tail split.

In [ ]:
pop = items['popularity'].sort_values(ascending=False).reset_index(drop=True)
rank = np.arange(1, len(pop) + 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ax = axes[0]
ax.loglog(rank, pop.values, marker='.', linestyle='-', alpha=0.7, color='#08589e', markersize=3)
# Fit a power law: log(pop) = a - s * log(rank)
mask = pop > 0
logr = np.log(rank[mask])
logp = np.log(pop[mask].values)
slope, intercept = np.polyfit(logr, logp, 1)
fitted = np.exp(intercept + slope * np.log(rank))
ax.loglog(rank, fitted, '--', color='red', alpha=0.6, label=f'fit: slope={slope:.2f}')
ax.set_xlabel('item rank (log)')
ax.set_ylabel('#positive interactions (log)')
ax.set_title('Item popularity — Zipf log-log')
ax.legend()

ax = axes[1]
total = pop.sum()
cum_pct = pop.cumsum() / total
ax.plot(rank / len(pop), cum_pct.values, linewidth=2.5, color='#08589e')
ax.plot([0, 1], [0, 1], '--', color='gray', alpha=0.5, label='uniform (no skew)')

# Highlight head-20% / tail-80% split
head_cut = int(0.2 * len(pop))
head_share = cum_pct.iloc[head_cut - 1]
ax.axvline(0.2, color='red', alpha=0.3, linestyle='--')
ax.axhline(head_share, color='red', alpha=0.3, linestyle='--')
ax.text(0.22, head_share - 0.05, f'top 20% items\n→ {head_share:.0%} of interactions', fontsize=10, color='red')
ax.set_xlabel('fraction of items (sorted by popularity)')
ax.set_ylabel('cumulative fraction of interactions')
ax.xaxis.set_major_formatter(PercentFormatter(1.0))
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_title('Cumulative interactions vs item fraction')
ax.legend(loc='lower right')

fig.suptitle('Figure 3 — Strong Zipf long tail (head 20% ≈ {:.0%} of interactions)'.format(head_share), fontsize=12, y=1.02)
fig.tight_layout()
save_fig(fig, '03_item_popularity_zipf.png')
plt.show()

n_zero = (items['popularity'] == 0).sum()
print(f'fitted Zipf slope: {slope:.2f}  (classical Zipf ≈ -1.0)')
print(f'top-20% items capture: {head_share:.1%} of all positives')
print(f'cold items (0 positives): {n_zero:,} of {len(items):,} ({n_zero/len(items):.1%})')

## 6. Temporal density — motivates sequential modelling

MovieLens timestamps reflect when a user logged in and rated a *batch*
of films from memory — so many ratings cluster within minutes (a rating
ceremony, not a viewing session). The signal we actually care about is
**whether users return on multiple distinct days** — if yes, there is
a behavioural trajectory for SASRec to model.

In [ ]:
ts = pd.to_datetime(inter['ts'], unit='s')
daily = ts.dt.floor('D').value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ax = axes[0]
ax.plot(daily.index, daily.values, alpha=0.35, color='#08589e', linewidth=0.6, label='raw daily')
rolling = daily.rolling(window=14, min_periods=1).mean()
ax.plot(rolling.index, rolling.values, color='#d62728', linewidth=2, label='14-day rolling mean')
ax.set_xlabel('date')
ax.set_ylabel('#positive interactions per day')
ax.set_title('Temporal density (Apr 2000 – Mar 2003)')
ax.legend()

ax = axes[1]
inter_with_date = inter.assign(date=pd.to_datetime(inter['ts'], unit='s').dt.floor('D'))
distinct_days = inter_with_date.groupby('user_id')['date'].nunique()
ax.hist(distinct_days, bins=np.geomspace(1, distinct_days.max() + 1, 40), color='#08589e', edgecolor='black', linewidth=0.4)
ax.set_xscale('log')
ax.set_xlabel('#distinct active days per user (log)')
ax.set_ylabel('#users')
ax.set_title('Distinct active days per user')
med_days = distinct_days.median()
ax.axvline(med_days, color='red', linestyle='--', label=f'median: {med_days:.0f} days')
ax.legend()

fig.suptitle('Figure 4 — Most users rate in a single ceremony; the long tail of multi-day users is what SASRec leverages', fontsize=12, y=1.02)
fig.tight_layout()
save_fig(fig, '04_temporal_density.png')
plt.show()

print()
print(f'distinct active days per user — median: {med_days:.0f}, p25: {distinct_days.quantile(0.25):.0f}, p75: {distinct_days.quantile(0.75):.0f}, max: {distinct_days.max()}')
share_multi_session = (distinct_days >= 3).mean()
print(f'{share_multi_session:.1%} of users are active on ≥3 distinct days — a minority but a meaningful tail.')
print()
print('Honest reading: MovieLens-1M is dominated by single-session "rating ceremonies" (median 1 active day),')
print('so SASRec mainly captures **within-session** item-to-item semantics (genre / style clustering within a batch),')
print('not multi-day preference drift.  This explains why SASRec\u2019s Recall@10 (0.0570) lags Two-Tower\u2019s (0.0590) — there is no rich cross-session signal here.')
print('On a streaming-style dataset (e.g. Last.fm, Yoochoose), SASRec\u2019s margin over CF would be expected to widen substantially.')

## 7. Genre frequency — feature for content-based cold-start

Cold-start TF-IDF uses **genres** as content tokens. If genres are too
skewed (e.g. 90% drama) the TF-IDF features collapse. We want a moderate
spread — ML-1M has 18 genres.

In [ ]:
genres_flat = []
for ids in items['genres']:
    genres_flat.extend(ids.tolist())
freq = pd.Series(genres_flat).value_counts()
freq.index = [GENRE_NAME[i] for i in freq.index]

fig, ax = plt.subplots(figsize=(12, 5.5))
colors = plt.cm.viridis(np.linspace(0.15, 0.85, len(freq)))
bars = ax.barh(freq.index[::-1], freq.values[::-1], color=colors[::-1], edgecolor='black', linewidth=0.4)
for bar, v in zip(bars, freq.values[::-1]):
    ax.text(v + 20, bar.get_y() + bar.get_height() / 2, f'{v}', va='center', fontsize=9)
ax.set_xlabel('#movies tagged with this genre')
ax.set_title('Figure 5 — Genre frequency across 3 533 items (multi-label)')

fig.tight_layout()
save_fig(fig, '05_genre_frequency.png')
plt.show()

print()
avg_genres = items['genres'].apply(len).mean()
print(f'avg. genres per movie: {avg_genres:.2f}  (multi-label, hence TF-IDF works well)')
print(f'head genre ({freq.index[0]}): {freq.iloc[0]} movies  ({freq.iloc[0]/len(items):.1%} of catalog)')
print(f'tail genre ({freq.index[-1]}): {freq.iloc[-1]} movies — provides discriminative signal for niche-taste users')

## 8. Cold-start proxy — bottom-decile (least active) vs top-decile users

ML-1M's authors already filtered the dataset to ≥20 ratings/user, so we
have **no truly cold users in the data**. To stress-test what our
`cold_start` channel actually serves, we look at the bottom-decile (D1,
least active) users as a **proxy** for new-user behaviour: they
interacted just enough to leave a signal, but their preferences are not
well established. We compare them to top-decile (D10) power users.

In [ ]:
q_low = act['n_positives'].quantile(0.10)
q_high = act['n_positives'].quantile(0.90)
low_users = act[act['n_positives'] <= q_low].index
high_users = act[act['n_positives'] >= q_high].index

raw_pos = raw_ratings[raw_ratings['rating'] >= 4].copy()
raw_pos['decile'] = np.where(raw_pos['raw_user'].isin(low_users), 'low',
                              np.where(raw_pos['raw_user'].isin(high_users), 'high', 'mid'))

movies_meta = pd.read_csv(
    RAW / 'movies.dat', sep='::', engine='python', header=None,
    names=['raw_item', 'title', 'genres_str'], encoding='latin-1',
)
raw_pos = raw_pos.merge(movies_meta[['raw_item', 'genres_str']], on='raw_item', how='left')

def top_genres(df, top=10):
    bag = []
    for g in df['genres_str'].dropna():
        bag.extend(g.split('|'))
    return pd.Series(bag).value_counts().head(top)

low_top = top_genres(raw_pos[raw_pos.decile == 'low'])
high_top = top_genres(raw_pos[raw_pos.decile == 'high'])

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

ax = axes[0]
low_rating = raw_ratings[raw_ratings['raw_user'].isin(low_users)]['rating'].value_counts(normalize=True).sort_index()
high_rating = raw_ratings[raw_ratings['raw_user'].isin(high_users)]['rating'].value_counts(normalize=True).sort_index()
x = np.arange(1, 6)
w = 0.4
ax.bar(x - w/2, [low_rating.get(i, 0) for i in x], w, label=f'D1 (least active, n={len(low_users):,}, ≤{q_low:.0f} pos)', color='#fc8d59', edgecolor='black', linewidth=0.4)
ax.bar(x + w/2, [high_rating.get(i, 0) for i in x], w, label=f'D10 (most active, n={len(high_users):,}, ≥{q_high:.0f} pos)', color='#3182bd', edgecolor='black', linewidth=0.4)
ax.set_xticks(x)
ax.set_xlabel('rating')
ax.set_ylabel('fraction of that group\u2019s ratings')
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_title('Rating distribution: D1 (low) vs D10 (high)')
ax.legend(fontsize=9)

ax = axes[1]
all_genres = sorted(set(low_top.index) | set(high_top.index))
low_pct = pd.Series({g: low_top.get(g, 0) for g in all_genres}).sort_values(ascending=False)
low_pct = low_pct / low_pct.sum()
high_pct = pd.Series({g: high_top.get(g, 0) for g in all_genres}).reindex(low_pct.index)
high_pct = high_pct / high_pct.sum()
y = np.arange(len(low_pct))
ax.barh(y - 0.2, low_pct.values, 0.4, color='#fc8d59', label='D1', edgecolor='black', linewidth=0.4)
ax.barh(y + 0.2, high_pct.values, 0.4, color='#3182bd', label='D10', edgecolor='black', linewidth=0.4)
ax.set_yticks(y)
ax.set_yticklabels(low_pct.index)
ax.set_xlabel('share of that group\u2019s positive interactions')
ax.xaxis.set_major_formatter(PercentFormatter(1.0))
ax.invert_yaxis()
ax.set_title('Top genres: D1 vs D10 users')
ax.legend(loc='lower right')

# Cosine similarity between the two genre distributions (alignment with high-pop fallback)
vec_low = np.array([low_pct.get(g, 0) for g in all_genres])
vec_high = np.array([high_pct.get(g, 0) for g in all_genres])
cos_sim = float(vec_low @ vec_high / (np.linalg.norm(vec_low) * np.linalg.norm(vec_high) + 1e-12))

fig.suptitle(f'Figure 6 — D1 (low-activity) vs D10 (high-activity): genre cosine similarity = {cos_sim:.2f}', fontsize=12, y=1.02)
fig.tight_layout()
save_fig(fig, '06_coldstart_subpopulation.png')
plt.show()

print()
print(f'D1 users (bottom 10%): n={len(low_users):,}, ≤{q_low:.0f} positives each — proxy for newly-joined users')
print(f'D10 users (top 10%): n={len(high_users):,}, ≥{q_high:.0f} positives each')
print(f'genre-preference cosine similarity (D1 vs D10): {cos_sim:.3f}')
print('→ D1 users prefer roughly the same head genres (drama/comedy/action) → mean-popularity fallback in `cold_start.py` is a defensible default for truly new users.')
print('→ The differences are in the *tail* (D1 leans more towards mass-appeal genres; D10 explores more niches) — exactly what TF-IDF over genres captures.')

## 9. Long-tail coverage — Lorenz curve / Gini

We've already seen the head-20% concentration. Now we quantify it with
the **Gini coefficient** (0 = perfectly uniform, 1 = single item gets
everything) and look at what catalog *coverage* a recommender achieves
if it always serves the top-K popular items — the worst-case baseline
for our `popularity` channel.

In [ ]:
pop_sorted = items['popularity'].sort_values().values.astype(float)
n = len(pop_sorted)
cum = np.cumsum(pop_sorted)
cum_share = cum / cum[-1]
lorenz_x = np.arange(1, n + 1) / n

# Gini = 1 - 2 * area under Lorenz curve  (np.trapezoid in NumPy 2.x; trapz in 1.x)
_trapz = getattr(np, 'trapezoid', getattr(np, 'trapz', None))
gini = 1 - 2 * _trapz(cum_share, lorenz_x)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

ax = axes[0]
ax.fill_between(lorenz_x, cum_share, lorenz_x, color='#fc8d59', alpha=0.3, label='inequality area')
ax.plot(lorenz_x, cum_share, color='#d62728', linewidth=2, label=f'Lorenz curve (Gini={gini:.2f})')
ax.plot([0, 1], [0, 1], '--', color='gray', alpha=0.5, label='perfect equality')
ax.set_xlabel('fraction of items (sorted by popularity)')
ax.set_ylabel('cumulative share of interactions')
ax.xaxis.set_major_formatter(PercentFormatter(1.0))
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_title('Lorenz curve — interaction inequality across catalog')
ax.legend(loc='upper left')

ax = axes[1]
K_values = [10, 50, 100, 200, 500, 1000, 2000, 3533]
coverages = [k / len(items) for k in K_values]
ax.bar(range(len(K_values)), coverages, color='#3182bd', edgecolor='black', linewidth=0.4)
for i, (k, c) in enumerate(zip(K_values, coverages)):
    ax.text(i, c + 0.015, f'{c:.0%}', ha='center', fontsize=9)
ax.set_xticks(range(len(K_values)))
ax.set_xticklabels([str(k) for k in K_values])
ax.set_xlabel('size of popularity-only recommendation list')
ax.set_ylabel('catalog coverage')
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_title('Catalog coverage from popularity-only recall')

fig.suptitle(f'Figure 7 — High inequality (Gini={gini:.2f}) — popularity-only recall covers <10% even at K=200', fontsize=12, y=1.02)
fig.tight_layout()
save_fig(fig, '07_longtail_coverage.png')
plt.show()

print()
print(f'catalog: {len(items):,} items')
print(f'Gini coefficient: {gini:.3f}  (1.0 = single item monopolises everything)')
print(f'  ↳ popularity-only top-10 covers {10/len(items):.2%} of catalog → terrible diversity')
print(f'  ↳ popularity-only top-200 covers {200/len(items):.2%} → still poor')
print(f'  → motivates the *content* + *learned* channels in the multi-channel recall stack')

## 10. Take-aways — how this EDA informs the W2 design

| Finding | Design choice it justifies |
|---|---|
| Rating ≥ 4 keeps **57.5%** of raw ratings — strong positive signal, dropping 3-star "meh" cases | binarisation threshold of 4.0 |
| Activity spans **2+ orders of magnitude** (≤14 to ≥484 positives, p10 vs p90) | needs both head-friendly (popularity) and tail-friendly (content) channels |
| Zipf slope **−1.57**; top-20% items capture **72.9%** of interactions | popularity is a strong baseline but biased; debias re-ranking will matter for diversity |
| Median user is active on **1 day** (rating ceremony); 24.4% on ≥3 distinct days | SASRec captures within-session item semantics; cross-session signal is weak on ML-1M — this explains its modest gap vs Two-Tower |
| 18 genres, multi-label (avg **1.69** per movie) | TF-IDF over genres is a non-degenerate content representation for cold-start |
| D1 vs D10 genre cosine similarity **≈ 0.998** | head-genre preferences are uniform — mean-popularity fallback in `cold_start.py` is essentially free; TF-IDF earns its keep on **item-level** discrimination, not user-level |
| Gini **≈ 0.7**; popularity-only top-200 covers **<6%** of catalog | multi-channel fusion (RRF) is needed for production-style catalog coverage |

All 7 plots are exported to `experiments/results/eda/` and embedded in
the project README §5 (Datasets).